In [ ]:
from pathlib import Path
import re

def load_medieval_occitan_corpus(data_dir: Path) -> list[str]:
    """Load and clean medieval Occitan texts for tokenizer training."""
    texts = []
    
    for txt_file in data_dir.rglob("*.txt"):
        with open(txt_file, "r", encoding="utf-8") as f:
            content = f.read()
        
        # Basic cleaning for medieval text
        content = re.sub(r'<[^>]+>', '', content)  # Remove XML/HTML tags
        content = re.sub(r'\[.*?\]', '', content)   # Remove editorial notes
        content = re.sub(r'\s+', ' ', content).strip()
        
        if len(content) > 100:  # Skip very short fragments
            texts.append(content)
    
    print(f"Loaded {len(texts)} texts, ~{sum(len(t) for t in texts)/1e6:.2f}M chars")
    return texts

# Usage
corpus = load_medieval_occitan_corpus(Path("data/occitan_medieval"))



## BPE tokenizer training
1) from scratch
2) Guarantee Coverage - Esteban's vocabulary

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from pathlib import Path

# 1. Prepare your combined text corpus
corpus_files = [
    "data/garces_transcriptions.txt",      # Garces's 82M synthetic pairs + real labels
    "data/your_manuscript_ground_truth.txt", # Your 1372 manuscript transcriptions
    "data/medieval_occitan_literature.txt"   # Additional historical texts (optional)
]

# 2. Initialize a fresh tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel()  # Handles spaces, special chars, unknown chars gracefully

# 3. Train on the COMBINED corpus
trainer = BpeTrainer(
    vocab_size=120,  # Slightly larger than Garces's 73 to capture common digraphs/trigraphs
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[EOS]"],
    show_progress=True
)
tokenizer.train(corpus_files, trainer)

# 4. Save
tokenizer.save("occitan_medieval_htr_tokenizer.json")
print(f"✅ Tokenizer trained with vocab size: {len(tokenizer.get_vocab())}")

In [ ]:
import json
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

# Load Garces's tokenizer to extract his character set
with open("garces_tokenizer.json", "r", encoding="utf-8") as f:
    garces_data = json.load(f)
garces_chars = list(garces_data["model"]["vocab"].keys())

# Initialize fresh tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel()

# Train with Garces's characters as the initial alphabet seed
trainer = BpeTrainer(
    vocab_size=120,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[EOS]"],
    initial_alphabet=garces_chars,  # ← Guarantees these characters are in the base vocab
    show_progress=True
)
tokenizer.train(corpus_files, trainer)
tokenizer.save("occitan_seeded_tokenizer.json")

## **To connect to the TrOCR**

In [ ]:
from transformers import AutoTokenizer, TrOCRProcessor, VisionEncoderDecoderModel

# Load your custom tokenizer
tokenizer = AutoTokenizer.from_pretrained("occitan_medieval_htr_tokenizer.json")

# Initialize TrOCR processor (handles image resizing + tokenizer)
processor = TrOCRProcessor(
    tokenizer=tokenizer,
    feature_extractor=None  # TrOCR handles images internally
)

# Now your decoder will predict token IDs that match your Old Occitan vocabulary
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    "microsoft/trocr-base-handwritten",
    "bert-base-uncased"
)
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))  # Align model vocab with your tokenizer